# A2.1 — Le binning des spectres

Un spectre est une liste de pics `(m/z, intensité)` de longueur variable. Un MLP ou un algorithme de plus proches
voisins a besoin d'entrées de même taille : on découpe l'axe m/z en cases régulières, et chaque case reçoit
l'intensité des pics qui y tombent.

Code : `src/casmi/features.py` (`bin_spectrum` pour un spectre, `bin_spectra` pour une matrice creuse).

Cinq choix à justifier : la **plage** (`max_mz`), la **largeur** des cases, l'**agrégation** dans une case, la
**transformation** des intensités et la **normalisation**.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from casmi.clean import load_clean_meta
from casmi.config import SEED
from casmi.features import BIN_WIDTH, MAX_MZ, bin_index, bin_spectra, bin_spectrum, n_bins
from casmi.io import load_spectra, load_test

pd.set_option("display.width", 200)

# Échantillon de 50 000 spectres fiables tirés au hasard (donc répartis sur tous les blocs du fichier)
meta = load_clean_meta()
rng = np.random.default_rng(SEED)
row_ids = np.sort(rng.choice(np.flatnonzero(meta["keep"]), 50_000, replace=False))
train = load_spectra(row_ids).merge(meta[["row_id", "precursor_mz"]], on="row_id")
test = load_test()
groups = {"train (échantillon)": train, "test": test}


def all_peaks(d):
    return np.concatenate(d["ms2_mzs"].to_list()), np.concatenate(d["ms2_normalized_intensities"].to_list())

## 1. La plage : `max_mz`

In [ ]:
rows = []
for name, d in groups.items():
    mz, _ = all_peaks(d)
    rows.append({"données": name, "pics": len(mz), "m/z médian": np.median(mz).round(1), "m/z max": mz.max().round(1),
                 **{f"% pics > {x}": round((mz > x).mean() * 100, 2) for x in (500, 1000, 1500, 2000)}})
pd.DataFrame(rows).set_index("données")

Aucun pic du test visible ne dépasse 460, et seuls 1,3 % des pics du train dépassent 1 000 (de grosses molécules
et des adducts dimères). **`max_mz = 1000`** : on garde 98,7 % des pics du train sans allonger le vecteur pour
des pics que le test n'a pas.

## 2. Les intensités : beaucoup de très petits pics

In [ ]:
for name, d in groups.items():
    _, it = all_peaks(d)
    print(f"{name:20s} pics < 1 % du pic principal : {(it < 0.01).mean():.1%}, < 0,1 % : {(it < 0.001).mean():.1%}")

pd.DataFrame({"intensité": [1, 0.1, 0.01, 0.001], "racine carrée": np.sqrt([1, 0.1, 0.01, 0.001]).round(3)})

La plupart des pics sont minuscules. Sans transformation, un pic à 1 % pèse 100 fois moins que le pic principal
et compte à peine dans une similarité. La **racine carrée** réduit l'écart (1 % → 0,1) sans faire remonter le
bruit autant qu'un logarithme. C'est la transformation par défaut (`transform="sqrt"`), `"log"` et `"none"` restent
disponibles pour les comparer.

## 3. La largeur des cases

Pour chaque largeur : la taille du vecteur, le nombre de cases non nulles par spectre, et la part des pics qui
partagent leur case avec un autre pic (et sont donc fusionnés).

In [ ]:
rows = []
for name, d in groups.items():
    for w in [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]:
        X = bin_spectra(d["ms2_mzs"], d["ms2_normalized_intensities"], bin_width=w)
        n_kept = sum(int(((np.asarray(m) >= 0) & (np.asarray(m) < MAX_MZ)).sum()) for m in d["ms2_mzs"])
        rows.append({"données": name, "largeur (Da)": w, "cases": n_bins(w), "cases non nulles par spectre": round(X.nnz / X.shape[0], 1),
                     "% pics fusionnés": round((1 - X.nnz / n_kept) * 100, 1)})
widths = pd.DataFrame(rows).set_index(["données", "largeur (Da)"])
widths

Les pics sont souvent très proches : à 0,1 Da, 38 % des pics du train (45 % de ceux du test) tombent dans une case
déjà occupée. Même à 0,01 Da, c'est encore 8 % : il s'agit souvent du même fragment découpé en plusieurs pics.

**Choix : 0,1 Da**, soit 10 000 cases.
- Plus fin (0,01 Da), le vecteur fait 100 000 cases : la première couche d'un MLP à 2 048 neurones aurait
  200 millions de poids (800 Mo), trop lourd pour nos machines.
- Plus large (1 Da), on ne garde que la masse nominale : des fragments de même masse entière mais de formules
  différentes se confondent, et on perd la précision des appareils (timsTOF : quelques ppm, soit environ 0,002 Da à
  500 m/z, bien moins qu'une case de 0,1 Da).
- La largeur reste un paramètre : comparer 0,1 et 1 Da (et 0,01 Da pour les plus proches voisins, qui supportent les
  grands vecteurs creux) fera partie des expériences de A4.1.

**Agrégation : le maximum.** Quand plusieurs pics proches tombent dans la même case, c'est en général le même
fragment ; la somme gonflerait les cases où le bruit s'accumule. `agg="sum"` reste disponible.

**Normalisation : norme L2 = 1**, ce qui rend le produit scalaire de deux vecteurs égal à leur similarité cosinus
(utile pour les plus proches voisins de C).

## 4. Un spectre avant et après le binning

In [ ]:
example = test.iloc[(test["ms2_mzs"].map(len) - 60).abs().argmin()]  # un spectre du test d'environ 60 pics
mz, it = np.asarray(example["ms2_mzs"]), np.asarray(example["ms2_normalized_intensities"])
v = bin_spectrum(mz, it)
centers = (np.flatnonzero(v) + 0.5) * BIN_WIDTH
top = mz[np.argmax(it)]
window = (np.floor(top) - 0.5, np.floor(top) + 1.5)

fig, axes = plt.subplots(3, 1, figsize=(8, 8), gridspec_kw={"height_ratios": [1, 1, 1]})
ax = axes[0]
ax.vlines(mz, 0, it, color="#2a78d6", linewidth=1.2)
ax.set_title(f"Spectre brut : {len(mz)} pics (precursor_mz = {example['precursor_mz']:.2f})", loc="left", fontsize=11)
ax.set_ylabel("intensité")
ax = axes[1]
ax.vlines(centers, 0, v[v > 0], color="#eb6834", linewidth=1.2)
ax.set_title(f"Vecteur biné : {np.count_nonzero(v)} cases non nulles sur {len(v):,} (racine carrée, norme L2)".replace(",", " "), loc="left", fontsize=11)
ax.set_ylabel("valeur")
for ax in axes[:2]:
    ax.set_xlim(0, example["precursor_mz"] + 20)
    ax.set_xlabel("m/z")
ax = axes[2]
inside = (mz >= window[0]) & (mz < window[1])
edges = np.arange(window[0], window[1] + BIN_WIDTH / 2, BIN_WIDTH)
ax.vlines(edges, 0, 1, color="#d6d5cf", linewidth=0.8)
ax.vlines(mz[inside], 0, np.sqrt(it[inside]), color="#2a78d6", linewidth=2, label="pics bruts (racine)")
k = bin_index(mz[inside])
ax.bar((np.unique(k) + 0.5) * BIN_WIDTH, np.sqrt([it[inside][k == b].max() for b in np.unique(k)]), width=BIN_WIDTH * 0.9,
       color="#eb6834", alpha=0.35, label="case : maximum")
ax.set_xlim(*window)
ax.set_xlabel("m/z (zoom autour du pic principal ; traits gris = frontières des cases de 0,1 Da)")
ax.set_title("Zoom : les pics proches tombent dans la même case", loc="left", fontsize=11)
ax.legend(frameon=False, loc="upper right")
for ax in axes:
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
plt.tight_layout()

Le zoom montre aussi l'effet de bord du binning : deux pics de part et d'autre d'une frontière (132,99 et 133,01)
tombent dans deux cases voisines, même s'ils sont très proches.

## 5. La taille en mémoire

In [ ]:
start = time.perf_counter()
X = bin_spectra(train["ms2_mzs"], train["ms2_normalized_intensities"])
elapsed = time.perf_counter() - start
size_mb = (X.data.nbytes + X.indices.nbytes + X.indptr.nbytes) / 1e6
print(f"{X.shape[0]:,} spectres binés en {elapsed:.1f} s, {X.nnz / X.shape[0]:.0f} cases non nulles par spectre")
print(f"Matrice creuse : {size_mb:.0f} Mo, soit {size_mb / X.shape[0] * len(meta) / 1e3:.1f} Go pour tout le train")
print(f"En dense : {len(meta) * n_bins() * 4 / 1e9:.0f} Go pour tout le train")
print(f"Spectres sans aucun pic sous {MAX_MZ:.0f} : {(X.getnnz(axis=1) == 0).sum()}")

Une matrice creuse ne stocke que les cases non nulles : environ 50 fois moins de mémoire qu'une matrice dense. C'est
ce qu'on utilisera pour les matrices de A2.3.

## 6. Les pertes neutres (option)

En chimie, ce que la molécule *perd* (H₂O, CO₂…) est souvent aussi informatif que le fragment. `neutral_losses`
calcule `precursor_mz − m/z` pour chaque pic ; le résultat se bine avec la même fonction, pour former un second
vecteur. Son intérêt sera mesuré en A4.1.